In [1]:
import cv2
from sklearn.externals import joblib
from skimage.feature import hog
import numpy as np

/home/nishat/anaconda3/lib/python3.7/site-packages/sklearn/externals/joblib/__init__.py:15: DeprecationWarning: sklearn.externals.joblib is deprecated in 0.21 and will be removed in 0.23. Please import this functionality directly from joblib, which can be installed with: pip install joblib. If this warning is raised when loading pickled models, you may need to re-serialize those models with scikit-learn 0.21+.
  warnings.warn(msg, category=DeprecationWarning)


In [5]:
# Loading the model
clf = joblib.load("Test_model.pkl")

KeyError: 72

In [3]:
# Loading the image
im = cv2.imread("digit-reco-1-in.jpg")

# RGB to Gray conversion
im_gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)

# Applying Gaussian Blur
im_gray = cv2.GaussianBlur(im_gray, (5, 5), 0)

# Thresholding the image
ret, im_th = cv2.threshold(im_gray, 90, 255, cv2.THRESH_BINARY_INV)

# Finding contour in image
ctrs, hier = cv2.findContours(im_th.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Getting Bounding Boxes
rects = [cv2.boundingRect(ctr) for ctr in ctrs]

# using HOG to find digit in each of the bounding box
for rect in rects:
    # Drawing the rectangles
    cv2.rectangle(im, (rect[0], rect[1]), (rect[0] + rect[2], rect[1] + rect[3]), (0, 255, 0), 3)
    
    # Making region aroung the digits
    leng = int(rect[3])
    pt1 = int(rect[1] + rect[3] // 2 - leng // 2)
    pt2 = int(rect[0] + rect[2] // 2 - leng // 2)
    roi = im_th[pt1:pt1+leng, pt2:pt2+leng]
    
    # Resize the image
    roi = cv2.resize(roi, (28, 28), interpolation=cv2.INTER_AREA)
    roi = cv2.dilate(roi, (3, 3))
    
    # Calculating the HOG features
    roi_hog_fd = hog(roi, orientations=9, pixels_per_cell=(14, 14), cells_per_block=(1, 1), visualise=False)
    nbr = clf.predict(np.array([roi_hog_fd], 'float64'))
    cv2.putText(im, str(int(nbr[0])), (rect[0], rect[1]),cv2.FONT_HERSHEY_DUPLEX, 2, (0, 255, 255), 3)
# cv2.imshow("Resulting Image",im)
# cv2.waitKey()

/home/nishat/anaconda3/lib/python3.7/site-packages/skimage/feature/_hog.py:239: skimage_deprecation: Argument `visualise` is deprecated and will be changed to `visualize` in v0.16
  'be changed to `visualize` in v0.16', skimage_deprecation)


ValueError: X has 36 features per sample; expecting 784

In [4]:
rect = rects[0]
rectangle = cv2.rectangle(im, (rect[0],rect[1]), (rect[0]+rect[2],rect[1]+rect[3]) , (0,255,0), 3 )

leng = int(rect[3])
pt1 = int(rect[1] + rect[3] // 2 - leng // 2)
pt2 = int(rect[0] + rect[2] // 2 - leng // 2)
roi = im_th[pt1:pt1+leng, pt2:pt2+leng]

roi_res = cv2.resize(roi, (28,28) , interpolation=cv2.INTER_AREA)
roi_dilated = cv2.dilate(roi_res, (3,3))

roi_hog_fd = hog(roi_dilated, orientations=9, pixels_per_cell=(14,14), cells_per_block=(1,1), visualize=False)
roi_hog_fd.shape

(36,)

In [29]:
cv2.waitKey(1)

233